# Lần thử cuối — luồng mô tả tự do (Colab GPU)

Một lần chạy: train 2 model trên cặp đúng dạng bài toán → đo toàn bộ lưới trên **dev** → chọn theo quy tắc đã chốt → chạy **test** một lần.

| thành phần | lựa chọn |
|---|---|
| dense | e5-base (hiện tại) / bge-m3 dùng nguyên / e5-base fine-tune |
| reranker | bge-reranker-base (hiện tại) / bge-reranker-base fine-tune |
| trộn | blend@0.3 (hiện tại) / blend@0.5 / replace |

Quy tắc chọn (trong `pipeline/final_eval.py`): trong các tổ hợp mà R@10 **không thấp hơn cấu hình hiện tại có ý nghĩa** ở cả query có dấu lẫn không dấu, chọn tổ hợp có **MRR@10 trung bình hai bản** cao nhất; không ai hơn thì giữ hiện tại.

Dữ liệu train: 2.487 issue (4.932 dòng, có bản bỏ dấu), đã loại mọi issue của bộ đo và issue gần trùng. Siêu tham số cố định trong `pipeline/train_task_models.py` (1 epoch).

**Trước khi chạy**
1. *Runtime → Change runtime type → T4 GPU* (hoặc tốt hơn nếu có).
2. Chuẩn bị `colab/final_bundle.zip`.
3. **Dữ liệu sẽ rời máy sang Google Colab.** Văn bản đã scrub nhưng vẫn còn tên service, mã lỗi, luồng nghiệp vụ.

Thời gian ước tính trên T4: train ~20–30 phút, dev ~30–45 phút, test ~40–60 phút.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "sentence-transformers==6.0.1" datasets accelerate

In [ ]:
from google.colab import files
up = files.upload()            # chon final_bundle.zip
BUNDLE = next(iter(up))
!unzip -oq {BUNDLE} -d /content/work
%cd /content/work
!ls pipeline data/processed

## 0. Chạy thử code train (5 bước mỗi model, ~2 phút)

Máy local thiếu `datasets`/`accelerate` nên code train được kiểm tra ở đây. Nếu ô này lỗi, dừng lại và gửi lỗi.

In [ ]:
!python pipeline/train_task_models.py --model bi --out /tmp/smoke-bi --max-steps 5 --limit 200
!python pipeline/train_task_models.py --model ce --out /tmp/smoke-ce --max-steps 5 --limit 50

## 1. Train

In [ ]:
!python pipeline/train_task_models.py --model bi --out models/e5-base-task
!python pipeline/train_task_models.py --model ce --out models/bge-reranker-base-task
!ls models

## 2. Dev: đo toàn bộ lưới và chọn

Kiểm tra nhanh trong bảng: `e5|bge|0.3` phải gần R@10 **0,626** (có dấu) và **0,568** (không dấu) như lần chọn cách trộn trước; `keyword` có dấu **0,586**.

In [ ]:
!python pipeline/final_eval.py --split dev --device cuda
!cat data/processed/final_choice.json | head -5

## 3. Test: chạy MỘT lần với lựa chọn đã ghi

In [ ]:
!python pipeline/final_eval.py --split test --device cuda

## 4. Tải kết quả về

File kết quả nhỏ. Model chỉ cần giữ nếu lựa chọn dùng model fine-tune (mỗi model ~1 GB); ô cuối tự nén đúng model được chọn và lưu sang Google Drive.

In [ ]:
!zip -q final_results.zip data/processed/final_eval_dev.json data/processed/final_eval_test.json data/processed/final_choice.json models/*/task_meta.json
files.download("final_results.zip")

In [ ]:
import json
chosen = json.load(open("data/processed/final_choice.json"))["chosen"]
dense, rr, _ = chosen.split("|")
need = [p for p, used in (("models/e5-base-task", dense == "e5-task"), ("models/bge-reranker-base-task", rr == "ce-task")) if used]
print("lua chon:", chosen, "| model can giu:", need or "khong (dung model cong khai)")
if need:
    from google.colab import drive
    drive.mount("/content/drive")
    for p in need:
        name = p.split("/")[-1]
        !zip -qr /content/drive/MyDrive/{name}.zip {p}
    print("da luu vao Google Drive: MyDrive/<ten model>.zip")